# 25. The Master EDA & Feature Engineering Decision Playbook

The definitive interactive decision playbook answering every practical question when facing new datasets and ML pipelines.


# The Master Decision Playbook

This playbook provides actionable answers to the **14 core practitioner questions** in Exploratory Data Analysis and Feature Engineering.

---

### Q1: "I have a numerical feature. What should I check?"
1. **Missingness %**: Is it MCAR (< 5%) or MNAR (informative)? $\rightarrow$ Add `is_missing` indicator if informative.
2. **Skewness & Tails**: If skew $> 1.0$ and $x > 0 \rightarrow$ Apply `np.log1p(x)`.
3. **Outlier Validity**: Is it a corrupt value (e.g. negative age) or legitimate tail? $\rightarrow$ Drop errors; winsorize or log-transform legitimate extremes.
4. **Zero Inflation**: If $> 30\%$ zeros $\rightarrow$ Create binary flag `is_zero` + continuous transform.

---

### Q2: "I have a categorical feature. What should I check?"
1. **Cardinality**:
   - Low ($\le 10$ levels) $\rightarrow$ **One-Hot Encoding**.
   - Medium ($11 - 30$ levels) $\rightarrow$ Consolidate rare levels ($< 1\%$) $\rightarrow$ **One-Hot Encoding**.
   - High ($> 30$ levels) $\rightarrow$ **Smoothed Target Encoding** or **Frequency Encoding**.
2. **Natural Order**: True hierarchy (Low < Med < High) $\rightarrow$ **Ordinal Encoding**.

---

### Q3: "I have two highly correlated features ($|r| > 0.85$). Should I remove one?"
- **For Linear / Logistic Regression**: YES $\rightarrow$ Collinearity inflates standard errors and flips coefficient signs. Drop one or engineer a composite ratio ($A / B$).
- **For Tree-Based Models (XGBoost/LightGBM)**: NO $\rightarrow$ Trees handle collinearity naturally by choosing the best split.

---

### Q4: "I am using XGBoost. Do I need scaling or log-transforms?"
- **Scaling**: NO $\rightarrow$ Decision trees split on rank order; multiplying a feature by $1,000$ or shifting by $10$ does not change split points.
- **Ratios & Domain Aggregations**: YES $\rightarrow$ Trees struggle to learn continuous division ($A / B$) across orthogonal splits.

---

### Q5: "I am using KNN or Logistic Regression. Do I need scaling?"
- **YES (MANDATORY)** $\rightarrow$ Unscaled large features completely dominate Euclidean distance and gradient steps.

---

### Q6: "Can I calculate this feature before the train/test split?"
- **RULE**: If the calculation requires calculating $\mu$, $\sigma$, median, target mean, or lookahead window across other rows $\rightarrow$ **NEVER**. Preprocessing must fit strictly on `X_train` inside a `Pipeline`.


In [ ]:
import pandas as pd
import numpy as np

# Interactive Decision Helper Function
def eda_fe_decision_advisor(dtype, skewness=0.0, cardinality=0, has_temporal=False, model_type='linear'):
    print("=== EDA & FE DECISION ADVISOR ===")
    print("Input: Type=", dtype, "| Skew=", skewness, "| Card=", cardinality, "| Model=", model_type)
    print("---------------------------------")
    
    if dtype == 'numerical':
        if skewness > 1.2:
            print("1. [TRANSFORM]: Strong right skew detected -> Apply np.log1p(x) or Yeo-Johnson.")
        if model_type in ['linear', 'knn', 'svm', 'nn']:
            print("2. [SCALING]: Distance/Gradient model -> Apply StandardScaler() or RobustScaler().")
        else:
            print("2. [SCALING]: Tree-based model -> Scaling is NOT required.")
    elif dtype == 'categorical':
        if cardinality <= 10:
            print("1. [ENCODING]: Low cardinality -> Use OneHotEncoder(drop='first' if linear).")
        else:
            print("1. [ENCODING]: High cardinality -> Use TargetEncoder(cv=5, smooth='auto') or FrequencyEncoding.")
            
    if has_temporal:
        print("3. [TEMPORAL]: Enforce Chronological Split; use .shift(1) before any rolling window calculation.")
    print("=================================")

# Example consultation
eda_fe_decision_advisor(dtype='numerical', skewness=2.4, model_type='linear', has_temporal=True)
